# Central Dogma of Molecular Biology

This notebook demonstrates the flow **coding DNA → mRNA → protein** using a curated coding sequence (CDS) for the human **DECR1** gene from NCBI Nucleotide.

A CDS is used rather than an arbitrary nucleotide record because a genomic region may contain introns or non-coding sequence and cannot necessarily be translated directly.

## 1. Install dependencies

Install the packages from `requirements.txt` in your environment before running the notebook:

```bash
python -m pip install -r requirements.txt
```

You do not need to run a package-install command inside the notebook.

In [ ]:
from Bio import Entrez, SeqIO
from Bio.Seq import Seq

# NCBI asks users to provide a valid contact email.
Entrez.email = "your.email@example.com"  # Replace with your email address


## 2. Retrieve a DECR1 coding sequence from NCBI

In [ ]:
# Search for human DECR1 mRNA records. We will inspect their CDS annotations.
search_term = 'DECR1[Gene] AND Homo sapiens[Organism] AND mRNA[Title]'

with Entrez.esearch(db="nucleotide", term=search_term, retmax=10) as handle:
    search_results = Entrez.read(handle)

record_ids = search_results["IdList"]
if not record_ids:
    raise RuntimeError("No matching DECR1 mRNA records were found. Check the search term or try again later.")

records = []
with Entrez.efetch(db="nucleotide", id=",".join(record_ids), rettype="gb", retmode="text") as handle:
    records = list(SeqIO.parse(handle, "genbank"))

# Pick the longest annotated CDS associated with DECR1 where possible.
candidates = []
for record in records:
    for feature in record.features:
        if feature.type != "CDS":
            continue
        gene_names = feature.qualifiers.get("gene", [])
        if gene_names and "DECR1" not in gene_names:
            continue
        cds_seq = feature.extract(record.seq)  # Correctly handles feature strand/orientation
        if len(cds_seq) >= 3 and len(cds_seq) % 3 == 0:
            candidates.append((len(cds_seq), record, feature, cds_seq))

if not candidates:
    raise RuntimeError("No suitable DECR1 CDS was found in the returned records.")

_, source_record, cds_feature, coding_dna = max(candidates, key=lambda item: item[0])
print("NCBI accession:", source_record.id)
print("Record description:", source_record.description)
print("CDS length (nt):", len(coding_dna))
print("CDS qualifiers:", {key: cds_feature.qualifiers[key] for key in ("gene", "product") if key in cds_feature.qualifiers})
print("Coding DNA sequence:", coding_dna)

## 3. Transcribe coding DNA into mRNA

Transcription replaces thymine (`T`) with uracil (`U`). The CDS extracted from the GenBank feature is already in coding orientation; `feature.extract()` accounts for strand orientation.

In [ ]:
messenger_rna = coding_dna.transcribe()
print("mRNA sequence:", messenger_rna)
assert "T" not in str(messenger_rna), "RNA should contain U rather than T."

## 4. Translate mRNA into protein

Translation reads the sequence in codons (groups of three nucleotides). Because this is an annotated CDS, its length should be divisible by three. The translated sequence may end in `*`, which represents a stop codon.

In [ ]:
protein = messenger_rna.translate(to_stop=False)
print("Protein sequence:", protein)
assert len(messenger_rna) % 3 == 0, "CDS length should be divisible by three."
assert str(coding_dna.translate(to_stop=False)) == str(protein), "DNA and RNA translations should match."

## 5. Check the sequence lengths

The coding DNA and mRNA have the same nucleotide length. Translation produces approximately one amino acid per three nucleotides, with the stop codon represented by `*` when included.

In [ ]:
print(f"Coding DNA: {len(coding_dna)} nucleotides")
print(f"mRNA:       {len(messenger_rna)} nucleotides")
print(f"Protein:    {len(protein)} amino-acid symbols (including a possible stop symbol)")

## Notes

- `padded_sequence` was removed because padding a biological coding sequence with `N` bases is not needed here and can distort translation.
- The RNA sequence is not reverse-complemented and relabelled as DNA. Reverse complementation is a strand-orientation operation, not a step in the central dogma.
- This notebook retrieves data from NCBI, so an internet connection is required. NCBI records and search results can change over time.
- Replace `your.email@example.com` with a valid email address before running the retrieval cell.